# Week 1 — BRFSS data check

In [1]:
import zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display


zip_path = "csv.zip"

with zipfile.ZipFile(zip_path) as z:
    files = {Path(name).name: name for name in z.namelist()
             if name.lower().endswith(".csv")}

years = range(2005, 2025)

## Read files

In [3]:
# Store summaries
overview = []
missing_rows = []
category_rows = []
columns_by_year = {}

features_to_check = [
    "diabetes", "income_00", "income_01", "income_level_00",
    "income_level_01", "race_00", "employment_status_00",
    "has_personal_doctor_00", "high_bp_00", "high_cholesterol_00"
]

with zipfile.ZipFile(zip_path) as z:
    for year in years:
        filename = files[f"DATASET_{year}.csv"]
        df = pd.read_csv(z.open(filename), low_memory=False)
        columns_by_year[year] = set(df.columns)
        overview.append({"year": year, "rows": len(df), "columns": len(df.columns)})

        # A missing percentage is computed
        for feature in df.columns:
            count = int(df[feature].isna().sum())
            missing_rows.append({"year": year, "feature": feature,
                                 "missing_count": count,
                                 "missing_pct": round(100 * count / len(df), 2)})

         # Save category counts, including missing
        for feature in features_to_check:
            if feature in df.columns:
                for value, count in df[feature].value_counts(dropna=False).items():
                    category_rows.append({"year": year, "feature": feature,
                                          "value": "<missing>" if pd.isna(value) else str(value),
                                          "count": int(count)})

overview = pd.DataFrame(overview)
missing = pd.DataFrame(missing_rows)
categories = pd.DataFrame(category_rows)
display(overview)
del df

,year,rows,columns
0,2005,355446,33
1,2006,354497,31
2,2007,429942,33
3,2008,412898,31
4,2009,431636,33
5,2010,449156,31
6,2011,503846,33
7,2012,473243,31
8,2013,490368,33
9,2014,461879,31


In [21]:
print("Between 10 to 20 Percent Missing Overview")
missing[missing.missing_pct.between(10, 20)

Between 10 to 20 Percent Missing Overview


,year,feature,missing_count,missing_pct
12,2005,income_level_00,48611,13.68
13,2005,income_00,48611,13.68
24,2005,high_cholesterol_00,66368,18.67
45,2006,income_level_00,49751,14.03
46,2006,income_00,49751,14.03
76,2007,income_level_00,58839,13.69
77,2007,income_00,58839,13.69
88,2007,high_cholesterol_00,67097,15.61
109,2008,income_level_00,53464,12.95
110,2008,income_00,53464,12.95


In [29]:
print("Between 20 to 50 Percent Missing Overview")
missing[missing.missing_pct.between(20, 50)]

Between 20 to 50 Percent Missing Overview


,year,feature,missing_count,missing_pct
18,2005,poor_health_days_00,170482,47.96
51,2006,poor_health_days_00,172876,48.77
82,2007,poor_health_days_00,210488,48.96
115,2008,poor_health_days_00,199528,48.32
146,2009,poor_health_days_00,211705,49.05
179,2010,poor_health_days_00,219988,48.98
210,2011,poor_health_days_00,247464,49.12
243,2012,poor_health_days_00,230368,48.68
274,2013,poor_health_days_00,242192,49.39
338,2015,poor_health_days_00,218430,49.66


In [31]:
print("Over 50 Percent Missing Overview")
missing[missing.missing_pct >= 50]

Over 50 Percent Missing Overview


,year,feature,missing_count,missing_pct
22,2005,avg_drinks_p_day_00,182983,51.48
55,2006,avg_drinks_p_day_00,188203,53.09
86,2007,avg_drinks_p_day_00,229206,53.31
119,2008,avg_drinks_p_day_00,220208,53.33
150,2009,avg_drinks_p_day_00,235939,54.66
183,2010,avg_drinks_p_day_00,246348,54.85
214,2011,avg_drinks_p_day_00,265968,52.79
247,2012,avg_drinks_p_day_00,248055,52.42
278,2013,avg_drinks_p_day_00,259761,52.97
307,2014,poor_health_days_00,231960,50.22


## Diabetes classes and feature availability

In [13]:
diabetes = categories[categories["feature"] == "diabetes"]
counts = diabetes.pivot(index="year", columns="value", values="count").fillna(0).astype(int)
print("Diabetes class counts:")
display(counts)
print("Diabetes class percentages:")
display(counts.div(overview.set_index("year")["rows"], axis=0).mul(100).round(2))

# Presence checks whether a column exists, independently of its missing rate.
all_features = sorted(set().union(*columns_by_year.values()))
presence = pd.DataFrame({year: [feature in columns_by_year[year]
                                for feature in all_features]
                         for year in years}, index=all_features)
print("Features not present in every year:")
display(presence.loc[~presence.all(axis=1)].replace({True: "Yes", False: "No"}))

Diabetes class counts:


value,Diabetic,Non-diabetic,Pre-diabetic
year,,,
2005,33317,318276,3853
2006,36080,313988,4429
2007,48068,375981,5893
2008,47391,360053,5454
2009,52381,372446,6809
2010,57464,384063,7629
2011,62456,433212,8178
2012,59749,405759,7735
2013,62341,419431,8596


Diabetes class percentages:


value,Diabetic,Non-diabetic,Pre-diabetic
year,,,
2005,9.37,89.54,1.08
2006,10.18,88.57,1.25
2007,11.18,87.45,1.37
2008,11.48,87.20,1.32
2009,12.14,86.29,1.58
2010,12.79,85.51,1.70
2011,12.40,85.98,1.62
2012,12.63,85.74,1.63
2013,12.71,85.53,1.75


Features not present in every year:


,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
high_bp_00,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No
high_cholesterol_00,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No
income_00,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,No,No,No
income_01,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,Yes,Yes,Yes
income_level_00,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No,No,No,No
income_level_01,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,Yes,Yes,Yes


## Inspect features

In [33]:
focus = ["employment_status_00", "avg_drinks_p_day_00", "poor_health_days_00",
         "bmi_00", "race_00", "income_00", "income_01",
         "high_bp_00", "high_cholesterol_00"]
missing_table = missing.pivot(index="feature", columns="year", values="missing_pct")
display(missing_table.reindex(focus))

def show_categories(feature, selected_years):
    return categories[(categories["feature"] == feature) &
                      (categories["year"].isin(selected_years))][
                          ["year", "value", "count"]].sort_values(["year", "value"])

print("Income categories, 2020–2021:")
display(show_categories("income_00", [2020, 2021]))
display(show_categories("income_01", [2020, 2021]))
print("Race categories, 2021–2023:")
display(show_categories("race_00", [2021, 2022, 2023]))
print("Employment, 2023–2024:")
display(show_categories("employment_status_00", [2023, 2024]))
print("Personal doctor, 2020–2022:")
display(show_categories("has_personal_doctor_00", [2020, 2021, 2022]))
print("High BP and cholesterol availability:")
display(presence.loc[["high_bp_00", "high_cholesterol_00"]]
        .replace({True: "Yes", False: "No"}))

year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
feature,,,,,,,,,,,,,,,,,,,,
employment_status_00,0.29,0.33,0.36,0.38,0.44,0.50,0.51,0.54,0.68,1.06,0.85,0.81,0.81,1.11,1.51,1.68,1.82,2.40,1.73,100.00
avg_drinks_p_day_00,51.48,53.09,53.31,53.33,54.66,54.85,52.79,52.42,52.97,53.56,53.04,52.09,51.79,52.06,53.32,53.25,52.78,53.18,51.53,54.49
poor_health_days_00,47.96,48.77,48.96,48.32,49.05,48.98,49.12,48.68,49.39,50.22,49.66,49.40,48.24,47.89,46.64,50.90,47.94,43.83,43.07,42.40
bmi_00,4.60,4.90,4.60,4.49,4.57,4.73,4.84,4.75,4.80,6.00,7.66,7.54,7.49,7.43,8.06,9.55,10.03,10.19,8.77,8.59
race_00,0.96,1.06,0.98,1.05,1.10,1.41,1.20,1.31,1.73,1.69,1.67,1.77,1.93,1.91,2.10,2.20,2.45,3.11,2.17,1.96
income_00,13.68,14.03,13.69,12.95,13.52,14.29,14.53,14.02,14.48,15.30,17.95,16.63,16.63,17.31,18.98,19.73,NaN,NaN,NaN,NaN
income_01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.39,21.33,19.88,18.83
high_bp_00,0.16,NaN,0.18,NaN,0.20,NaN,0.26,NaN,0.27,NaN,0.29,NaN,0.28,NaN,0.34,NaN,0.38,NaN,0.40,NaN
high_cholesterol_00,18.67,NaN,15.61,NaN,13.57,NaN,14.67,NaN,14.53,NaN,14.19,NaN,7.05,NaN,6.91,NaN,14.52,NaN,12.65,NaN


Income categories, 2020–2021:


,year,value,count
712,2020,1.0,12865
711,2020,2.0,13669
710,2020,3.0,20993
709,2020,4.0,27649
708,2020,5.0,31346
707,2020,6.0,43771
706,2020,7.0,52381
704,2020,8.0,117662
705,2020,<missing>,78714


,year,value,count
758,2021,1.0,10840
754,2021,10.0,19744
755,2021,11.0,18924
757,2021,2.0,11501
756,2021,3.0,14916
753,2021,4.0,21016
752,2021,5.0,43803
749,2021,6.0,48277
748,2021,7.0,59346
750,2021,8.0,47787


Race categories, 2021–2023:


...

,year,value,count
892,2023,<missing>,7473
888,2023,A homemaker,17479
889,2023,A student,10206
884,2023,Employed for wages,177312
890,2023,Out of work for less that 1 year,8947
891,2023,Out of work for more than 1 year,7631
885,2023,Retired,139497
886,2023,Self-employed,37841
887,2023,Unable to work,25360
940,2024,<missing>,454774


Personal doctor, 2020–2022:


,year,value,count
743,2020,<missing>,1991
742,2020,More than one,27173
741,2020,No,69783
740,2020,"Yes, only one",300103
792,2021,<missing>,3572
790,2021,More than one,123975
791,2021,No,52551
789,2021,"Yes, only one",257152
847,2022,<missing>,4249
845,2022,More than one,136075


High BP and cholesterol availability:


,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
high_bp_00,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No
high_cholesterol_00,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No,Yes,No
